In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import random
from typing import Callable

MIN: int = 0
MAX: int = 4
BINS: int = 100
NUM_SAMPLES: int = 100000
N: int = 5

In [ ]:
# Toy language model that returns a uniformly distributed random number
def model(num_samples) -> np.ndarray:
    return np.random.uniform(0, MAX, num_samples)

def histogram(output: list[int]):
    hist, bins = np.histogram(output, bins=BINS, range=(MIN, MAX), density=False)
    probs = hist / np.sum(hist)
    return probs, bins

In [ ]:
# The ground truth reward model. We assume that we have a preference for the number `mid`.
def reward_model_ground_truth(output) -> float:
    # TODO
    return 5 - np.abs(2 - output)

# Definition of the proxy reward model. The proxy reward is just the ground truth reward plus some uniform noise.
def reward_model_proxy(output) -> float:
    # TODO
    return reward_model_ground_truth(output)

In [ ]:
def plot_rewards() -> None:
    outputs = np.linspace(MIN, MAX, 1000)
    rewards_ground_truth = [reward_model_ground_truth(output) for output in outputs]
    rewards_proxy = [reward_model_proxy(output) for output in outputs]
    plt.plot(outputs, rewards_ground_truth, alpha=1.0, label='R(x) = 5 - |2 - x|')
    plt.xlabel("output (x)")
    plt.ylabel("reward R(x)")
    plt.title("Problem 1(a): Reward Function")
    plt.legend()
    plt.grid(True, alpha=0.3)

# Plot the reward function for Problem 1(a)
plot_rewards()

In [ ]:
def best_of_n(n: int, reward_model):
    samples = model(n)
    rewards = [reward_model(sample) for sample in samples]
    best_idx = np.argmax(rewards)
    return samples[best_idx], rewards[best_idx]


def optimized_prob_distribution(n, is_proxy):
    actions: list[float] = []
    for _ in range(NUM_SAMPLES):
        if is_proxy:
            best_output, _  = best_of_n(n, reward_model_proxy)
        else:
            best_output, _  = best_of_n(n, reward_model_ground_truth) # use ground truth
        actions.append(best_output)
    probs, bins = histogram(actions)
    return probs, bins

# Probabilities before best-of-n sampling
probs_initial: list[int] = BINS * [1/BINS]

# Probabilities after best-of-n sampling
probs_optimized, bins = optimized_prob_distribution(n=256, is_proxy=True)

def plot_optimized_output() -> None:
    plt.hist(bins[:-1], bins, weights=probs_optimized)
    plt.xlabel("output")
    plt.ylabel("prob(output)")

# Plot the output after best-of-n sampling using the proxy reward model
plot_optimized_output()

In [ ]:
def estimate_reward(n:int, reward_model: Callable) -> float:
    # TODO
    num_runs = 100
    total_reward = 0
    for _ in range(num_runs):
        _, reward = best_of_n(n, reward_model)
        total_reward += reward
    return total_reward / num_runs

rewards_ground_truth: list[float] = []

RANGE_N: list[int] = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048]
for n in RANGE_N:
    reward_ground_truth: float = estimate_reward(n, reward_model_ground_truth)
    rewards_ground_truth.append(reward_ground_truth)

# Plot E[R(π_BoN)] vs n for Problem 1(b)
plt.plot(RANGE_N, rewards_ground_truth)
plt.xscale('log')
plt.ylabel('E[R(π_BoN)]')
plt.xlabel('n')
plt.title('Problem 1(b): Expected Reward vs N')
plt.legend(['ground truth'])
plt.show()

In [ ]:
# Observations for Problem 1
print("=" * 60)
print("OBSERVATIONS FOR PROBLEM 1")
print("=" * 60)
print("\nProblem 1(a):")
print("The reward function R(x) = 5 - |2 - x| is a V-shaped function")
print("centered at x = 2 with maximum reward of 5.")
print("Rewards decrease linearly as outputs move away from 2.")
print()
print("Problem 1(b):")
print(f"Sample rewards for different N values:")
for i, n in enumerate(RANGE_N):
    print(f"  N = {n:4d}: E[R(π_BoN)] = {rewards_ground_truth[i]:.4f}")
print()
print("Key observations:")
print("- As N increases, the expected reward E[R(π_BoN)] monotonically increases")
print("- Larger N values allow better selection of samples closer to the optimal x = 2")
print("- Diminishing returns: reward gains decrease as N grows larger")
print(f"- Maximum observed reward: {max(rewards_ground_truth):.4f} (approaching theoretical max of 5)")
print()